In [ ]:
# Komórka 1: Importy wymaganych bibliotek
import pandas as pd
import numpy as np
import os
import boto3
from io import BytesIO
from dotenv import load_dotenv
from pycaret.regression import *


In [ ]:
# Komórka 2: Konfiguracja połączenia z chmurą
load_dotenv()

BUCKET_NAME = "halfmaratonwroclaw"
s3 = boto3.client(
    "s3",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    endpoint_url=os.getenv("AWS_ENDPOINT_URL_S3"),
)

# Test połączenia i uprawnień
try:
    s3.list_buckets()  # lub s3.head_bucket(Bucket=BUCKET_NAME)
    print("Połączenie z S3 działa poprawnie")
except Exception as e:
    print(f"Błąd połączenia z S3: {str(e)}")

Błąd połączenia z S3: An error occurred (AccessDenied) when calling the ListBuckets operation: Access Denied.


In [ ]:
# Komórka 3: Definicje funkcji pomocniczych
def convert_time_to_seconds(time):
    """Konwersja czasu na sekundy."""
    if pd.isnull(time) or time in ['DNS', 'DNF']:
        return None
    try:
        parts = time.split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
    except:
        return None

def upload_file_to_s3(file_path, s3_path):
    """Wysyłanie pliku do chmury."""
    try:
        s3.upload_file(Filename=file_path, Bucket=BUCKET_NAME, Key=s3_path)
        print(f"Pomyślnie wysłano {file_path} do {s3_path}")
    except Exception as e:
        print(f"Błąd podczas wysyłania pliku {file_path}: {str(e)}")

def read_local_files(data_folder="Pliki z danymi"):
    """
    Odczytuje pliki CSV z lokalnego folderu.
    
    Args:
        data_folder: Nazwa folderu z danymi (domyślnie 'Pliki z danymi')
    
    Returns:
        Lista dataframe'ów z wczytanymi danymi
    """
    dfs = []
    try:
        if not os.path.exists(data_folder):
            raise FileNotFoundError(f"Folder '{data_folder}' nie istnieje!")
        
        for file in os.listdir(data_folder):
            if file.endswith('.csv'):
                file_path = os.path.join(data_folder, file)
                print(f"Wczytuję plik: {file}")
                df = pd.read_csv(file_path, sep=';')
                dfs.append(df)
                
                # Wysyłanie pliku do chmury
                s3_path = f"stocks/{file}"
                upload_file_to_s3(file_path, s3_path)
                
        if not dfs:
            raise FileNotFoundError("Nie znaleziono plików CSV w podanym folderze!")
            
        return dfs
    
    except Exception as e:
        print(f"Wystąpił błąd podczas wczytywania plików: {str(e)}")
        return None

def remove_outliers(df, columns):
    """
    Usuwanie wartości odstających metodą IQR dla wybranych kolumn.
    
    Args:
        df: DataFrame z danymi
        columns: Lista kolumn do sprawdzenia outlierów
        
    Returns:
        DataFrame bez wartości odstających
    """
    df_clean = df.copy()
    
    for column in columns:
        Q1 = df_clean[column].quantile(0.25)
        Q3 = df_clean[column].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers_mask = ~((df_clean[column] < lower_bound) | (df_clean[column] > upper_bound))
        df_clean = df_clean[outliers_mask]
        
        print(f"Usunięto {len(df) - len(df_clean)} outlierów z kolumny {column}")
        
    return df_clean

def clean_data(df):
    """Czyszczenie i przygotowanie danych."""
    # Konwersja kolumn czasowych
    time_columns = ['Czas', '5 km Czas', '10 km Czas', '15 km Czas', '20 km Czas']
    for col in time_columns:
        if col in df.columns:
            df[col] = df[col].apply(convert_time_to_seconds)
    
    # Usuwanie niepotrzebnych kolumn
    columns_to_drop = [
        'Miejsce', 'Numer startowy', 'Płeć Miejsce', 'Kategoria wiekowa Miejsce',
        'Imię', 'Nazwisko', 'Miasto', 'Kraj', '5 km Miejsce Open', 
        '10 km Miejsce Open', '15 km Miejsce Open', '20 km Miejsce Open'
    ]
    df = df.drop(columns_to_drop, axis=1)
    
    # Uzupełnienie brakujących wartości kategorycznych
    df[['Płeć', 'Kategoria wiekowa']] = df[['Płeć', 'Kategoria wiekowa']].fillna('Unknown')
    
    # Uzupełnienie brakujących wartości numerycznych
    numeric_columns = [
        'Rocznik', '5 km Czas', '5 km Tempo', '10 km Czas', '10 km Tempo',
        '15 km Czas', '15 km Tempo', '20 km Czas', '20 km Tempo',
        'Tempo Stabilność', 'Czas', 'Tempo'
    ]
    
    for col in numeric_columns:
        df[col] = df[col].fillna(df[col].median())
    
    # Usuwanie outlierów dla kluczowych kolumn
    columns_for_outliers = [
        'Rocznik',
        '5 km Czas',
        '10 km Czas',
        '15 km Czas',
        '20 km Czas',
        'Czas',
        'Tempo Stabilność'
    ]
    
    df = remove_outliers(df, columns_for_outliers)
    
    return df

def check_column_names(dfs):
    """Sprawdza czy wszystkie DataFrames mają te same nazwy kolumn."""
    first_df_columns = dfs[0].columns
    return all(df.columns.equals(first_df_columns) for df in dfs[1:])

In [ ]:
# Komórka 4: Wczytanie i przygotowanie danych
# Wczytanie lokalnych plików i wysłanie ich do chmury
print("Wczytywanie lokalnych plików...")
local_dfs = read_local_files()

if local_dfs is None or len(local_dfs) == 0:
    print("Błąd podczas wczytywania lokalnych plików!")
    exit()

# Sprawdzenie czy wszystkie ramki danych mają te same kolumny
print("\nSprawdzanie spójności kolumn...")
if not check_column_names(local_dfs):
    print("Ostrzeżenie: Pliki mają różne struktury kolumn!")

# Połączenie ramek danych
print("\nŁączenie danych...")
merged_df = pd.concat(local_dfs, ignore_index=True)

# Czyszczenie danych
print("\nCzyszczenie danych...")
prepared_df = clean_data(merged_df)

# Wyświetlenie podstawowych statystyk po czyszczeniu
print("\nPodstawowe statystyki po usunięciu outlierów:")
print(prepared_df.describe())

In [ ]:
# Komórka 5: Trenowanie modelu
# Inicjalizacja eksperymentu
exp = setup(
    data=prepared_df,
    target='Czas',
    session_id=123,
    verbose=False,
    fold=5
)

# Trenowanie i porównanie modeli
best_model = compare_models()
metrics_df = pull()

# Dostrajanie modelu
tuned_best_model = tune_model(best_model, n_iter=50, optimize='R2')
final_best_model = compare_models([best_model, tuned_best_model])

# Finalizacja modelu
my_final_best_model = finalize_model(final_best_model)

In [ ]:
# Komórka 6: Analiza wyników
# Wyświetlenie metryk
print("\nMetryki wydajności modelu:")
print(metrics_df)

# Wykresy
exp.plot_model(final_best_model, plot='feature')
exp.plot_model(final_best_model, plot='error')
exp.plot_model(final_best_model, plot='residuals')

# Zapisanie wartości granicznych tempa
max_tempo = prepared_df['Tempo'].max()
min_tempo = prepared_df['Tempo'].min()

print(f"\nWartości graniczne tempa po usunięciu outlierów:")
print(f"Maksymalne tempo: {max_tempo}")
print(f"Minimalne tempo: {min_tempo}")

In [ ]:
# Komórka 7: Zapisywanie modelu
# Utworzenie katalogu na model
os.makedirs('models', exist_ok=True)

# Zapisanie modelu lokalnie
save_model(final_best_model, 'models/my_final_model_pipeline')

# Wysłanie modelu do chmury
for root, dirs, files in os.walk("models"):
    for file in files:
        file_path = os.path.join(root, file)
        s3_path = f"halfmarathon/{file}"
        upload_file_to_s3(file_path, s3_path)
        print(f"Wysyłam plik {os.path.join(root, file)} do {s3_path}")